# 00 — Check learning and cost before freezing

Run **one seed-17 validation stream**. Start with CIFAR-10/platform, then learned; repeat for CIFAR-100. These are development observations, not confirmation results.

Select a **TensorFlow 2.20 / Keras 3** kernel, restart the kernel, then **Run All**.

In [ ]:
import os
import sys
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "semantic_consolidation/config.py").is_file())
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from notebooks.thesis.workflow import check_runtime
print(check_runtime())
from IPython import get_ipython
get_ipython().run_line_magic("matplotlib", "inline")
from common.dataloader import get_datasets
from common.model import get_model
from common.train import train_model
from notebooks.thesis.workflow import load_development, attach_route, close_run, finish_run
from notebooks.thesis.presentation import describe_run, show_learning_results, show_diagnostics, show_saved_replay

### 1. Select one stream

Only DATASET and CONDITION normally need changing. The [recipe rationale](HYPERPARAMETER_RATIONALE.md) explains the central YAML.

In [ ]:
DATASET = "cifar10"  # "cifar10" or "cifar100"
CONDITION = "baseline"  # Start here, then "learned" in a fresh kernel.
SEED = 17
config, context = load_development(ROOT / "notebooks/thesis/configs" / f"{DATASET}.yaml",
                                   condition=CONDITION, seed=SEED)
describe_run(config)

### 2. Load data and create the model

The common APIs own splitting, replay and class growth. The route attaches to this same model.

In [ ]:
project = config.common
trainset, valset = get_datasets(project)
bundle = get_model(project)
attach_route(context, bundle)

### 3. Train once

After an interruption, restart the kernel and **Run All** to resume the same stream. Work after the latest valid checkpoint is repeated.

In [ ]:
if context.get("training_started"):
    raise RuntimeError("Restart the kernel before training another stream.")
context["training_started"] = True
try:
    history = train_model(project, bundle, trainset, valset=valset)
except BaseException:
    close_run(context, release=True)
    raise
finally:
    close_run(context)

### 4. Save and read the results

Accuracy is percent; forgetting and backward transfer are signed percentage points. These are validation observations for development.

In [ ]:
evaluations = finish_run(context, config, bundle, history, trainset, valset)
RUN, VIEW_DIR = show_learning_results(config, bundle)

### 5. Review phases, cost and saved replay

Read the full stream before freezing: new-class learning, old-class retention, phase changes, gate visits and late-task cost. Replay labels are generation conditions, not verified image semantics.

In [ ]:
review = show_diagnostics(RUN, VIEW_DIR)
show_saved_replay(config, bundle, RUN, VIEW_DIR)

Before freezing, inspect new-class learning against the plotted all-seen chance level, old-class retention, replay, phase deltas and gate visits. Check the full ten-task CIFAR-100 run for late-task cost and memory. Tune only the central YAML if development evidence warrants it, record why, and rerun the necessary stream. Software smoke checks do not establish useful learning. Once the recipe is adequate, run **01_Freeze_Experiment.ipynb**.